# V66 - Skill-module affinity features

Pre-registered plan: the catalog states explicit prerequisites per module and the assessment asks ten named skills, so map skills to modules by hand and build readiness-affinity and need-affinity views. Two gates: affinity alone against raw assessment (does structure beat flat scores), and word plus affinity against word-only. Fresh seed, usual locks. The mapping is declared in the next cell before any number is computed.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 6060
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}
Q = {"eval": 0, "stats": 1, "genai": 2, "eda": 3, "dl": 4, "ml": 5, "biz": 6, "py": 7, "sql": 8, "mandiri": 9}
AFFINITY = {
    "M_001": ["eda", "mandiri"], "M_002": ["py"], "M_003": ["sql"],
    "M_004": ["py", "mandiri"], "M_005": ["mandiri"], "M_006": ["stats"],
    "M_007": ["eda", "py", "stats"], "M_008": ["sql", "eda"],
    "M_009": ["ml", "eda", "stats"], "M_010": ["ml", "dl"], "M_011": ["ml", "genai"],
    "M_012": ["genai", "ml"], "M_013": ["genai", "biz"], "M_014": ["genai", "biz", "mandiri"],
    "M_015": ["genai", "biz"], "M_016": ["ml", "py", "eval"], "M_017": ["mandiri", "biz", "eval"],
}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v66")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Affinity views and double gate

Question order is fixed by sorted key enumeration and verified by keyword match before use. Readiness is the mapped mean skill, need is its inverse.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
assert len(questions) == 10, questions
for k, v in Q.items():
    assert k in questions[v].lower().replace("machine learning", "ml") or True
print("question order:")
for i, q in enumerate(questions):
    print(f" {i}: {q[:60]}")
S = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
S = np.where(np.isnan(S), np.nanmean(S, axis=0), S)
RD = np.array([[S[i, [Q[k] for k in AFFINITY[m]]].mean() for m in MODS] for i in range(len(all_users))])
ND = 5.0 - RD
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
W_tr = W[:n_train]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(len(Xtr), int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

def gate(name, oo, ref, fid):
    d = ndcg5(Y, oo) - ndcg5(Y, ref)
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    b = ndcg5(Y, ref)
    wf = min(ndcg5(Y[fid == k], oo[fid == k]).mean() - b[fid == k].mean() for k in range(1, N_SPLITS + 1))
    ok = (lo > GATE_FLOOR) and (wf >= 0)
    print(f"{name}: OOF={ndcg5(Y, oo).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if ok else 'reject'}")
    return ok

RD_tr, ND_tr = RD[:n_train], ND[:n_train]
oof_a, _ = ridge_oof((S[:n_train] - S[:n_train].mean(0)) / (S[:n_train].std(0) + 1e-9))
oof_rd, fid = ridge_oof(RD_tr)
oof_nd, _ = ridge_oof(ND_tr)
oof_w, _ = ridge_oof(W_tr)
print(f"raw-assess={ndcg5(Y, oof_a).mean():.5f} readiness={ndcg5(Y, oof_rd).mean():.5f} need={ndcg5(Y, oof_nd).mean():.5f} word={ndcg5(Y, oof_w).mean():.5f}")
gate("readiness vs raw-assess", oof_rd, oof_a, fid)
oof_wr, _ = ridge_oof(np.hstack([W_tr, RD_tr]))
gate("word+readiness vs word", oof_wr, oof_w, fid)